# Explore the staged Green Taxi file (phase 2)
Exploring only. The pipeline never imports or runs this notebook.

In [ ]:
import os
from pathlib import Path

import duckdb

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
staging = root / os.environ.get("STAGING_DIR", "data/raw")
file = staging / "green_tripdata_2025-01.parquet"
assert file.exists(), f"missing {file}: run make download PARTITION=2025-01"
src = f"read_parquet('{file}')"
con = duckdb.connect()

## Columns and types

In [2]:
con.sql(f'describe select * from {src}').df()[['column_name', 'column_type']]

IOException: IO Error: No files found that match the pattern "data/raw/green_tripdata_2025-01.parquet"

## Row count (expected value for A1)

In [ ]:
con.sql(f'select count(*) as rows from {src}').df()

## Missing values in the columns the output needs, and in ehail_fee

In [ ]:
con.sql(f'''select count(*) filter (where lpep_pickup_datetime is null) as null_pickup,
  count(*) filter (where total_amount is null) as null_total_amount,
  count(*) filter (where ehail_fee is null) as null_ehail_fee
from {src}''').df()

## Value ranges

In [ ]:
con.sql(f'''select min(lpep_pickup_datetime) as first_pickup, max(lpep_pickup_datetime) as last_pickup,
  min(total_amount) as min_total, max(total_amount) as max_total, round(sum(total_amount), 2) as sum_total,
  count(*) filter (where total_amount < 0) as negative, count(*) filter (where total_amount = 0) as zero
from {src}''').df()

## Pickup dates outside the file's own month, and outside Jan to Mar 2025 (ADR 2)

In [ ]:
con.sql(f'''select cast(lpep_pickup_datetime as date) as pickup_date, count(*) as rows
from {src} where lpep_pickup_datetime < '2025-01-01' or lpep_pickup_datetime >= '2025-02-01'
group by 1 order by 1''').df()

## Natural key: is any row repeated on all columns?

In [ ]:
con.sql(f'''select count(*) as rows, (select count(*) from (select distinct * from {src})) as distinct_rows from {src}''').df()